In [1]:
!zcat ../../data/zymo/ERR3152366.fastq.gz | echo $((`wc -l`/4))

3667480


In [3]:
!cat ../../data/zymo/SRR13128014.fastq | echo $((`wc -l`/4))

1978852


In [4]:
!cat ../../data/environmental_samples/SRR14560391_1.fastq | echo $((`wc -l`/4))

21344959


In [3]:
!cat ../../data/SPMP/ERR7625321.fastq | echo $((`wc -l`/4))

1050196


In [8]:
import pandas as pd

dataset = ["Zymo Log, Nanopore GridION",
           "Zymo Gut, PacBio Hifi"]
read_name = ["ERR3152366", "SRR13128014"]
num_reads = [3667480, 1978852]
num_primary_maps = [3297526, 1918031]

percentage_classified_sylph = []
percentage_classified_skiver = []

for read in read_name:
    sylph_df = pd.read_csv(f"../../output/predict_unclassified_reads/{read}_sylph_u.tsv", sep="\t")
    percentage_classified_sylph.append(sylph_df["Sequence_abundance"].sum() / 100)
    skiver_df = pd.read_csv(f"../../output/predict_unclassified_reads/{read}_sylph_u_read_id.tsv", sep="\t")
    percentage_classified_skiver.append(skiver_df["Sequence_abundance"].sum() / 100)

results_df = pd.DataFrame({
    "Dataset": dataset,
    "Read Name": read_name,
    "Number of Reads": num_reads,
    "Number of Primary Maps": num_primary_maps,
    "Percentage Classified (Sylph)": percentage_classified_sylph,
    "Percentage Classified (Skiver)": percentage_classified_skiver
})

results_df["Percentage Classified (minimap2)"] = results_df["Number of Primary Maps"] / results_df["Number of Reads"]



In [9]:
results_df

,Dataset,Read Name,Number of Reads,Number of Primary Maps,Percentage Classified (Sylph),Percentage Classified (Skiver),Percentage Classified (minimap2)
0,"Zymo Log, Nanopore GridION",ERR3152366,3667480,3297526,0.345356,0.821668,0.899126
1,"Zymo Gut, PacBio Hifi",SRR13128014,1978852,1918031,0.960607,0.973736,0.969265
2,"SPMP, Nanopore PromethION",ERR7625321,1050196,942093,0.679725,0.795762,0.897064
3,"Environmental Sample, Illumina",SRR14560391,21344959,14129159,0.238220,0.218594,0.661944


In [1]:
3297526 / 3667480

0.8991258302703764

In [2]:
1918031 / 1978852

0.9692645028531695

In [5]:
14129159 / 21344959

0.6619435998916653

In [1]:
# Gather the contig name from the reference genome

import glob

zymo_mock_genomes = glob.glob("../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Genomes/*.fasta")
zymo_gut_genomes = glob.glob("../../data/zymo/D6331.refseq/genomes/*.fasta")

# For each line that starts with ">", extract the contig name and store it in a list

zymo_mock_contigs = {}
for genome in zymo_mock_genomes:
    with open(genome, "r") as f:
        for line in f:
            if line.startswith(">"):
                contig_name = line.split()[0][1:]
                zymo_mock_contigs[contig_name] = genome

zymo_gut_contigs = {}
for genome in zymo_gut_genomes:
    with open(genome, "r") as f:
        for line in f:
            if line.startswith(">"):
                contig_name = line.split()[0][1:]
                zymo_gut_contigs[contig_name] = genome

In [5]:
zymo_mock_contigs["BS.pilon.polished.v3.ST170922"]

'../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Genomes/Bacillus_subtilis_complete_genome.fasta'

In [2]:
import pandas as pd

zymo_log_map_df = pd.read_csv("../../output/zymo/ERR3152366.per_aln_stats.csv")
sylph_res_df = pd.read_csv("../../output/predict_unclassified_reads/ERR3152366_sylph_gt_u.tsv", sep="\t")

print(len(set(zymo_log_map_df["read"])), len(zymo_log_map_df))

for i in range(len(zymo_log_map_df)):
    if zymo_log_map_df.loc[i, "alignment_type"] != "primary":
        continue
    chr_name = zymo_log_map_df.loc[i, "chr"]
    if chr_name in zymo_mock_contigs:
        zymo_log_map_df.loc[i, "genome"] = zymo_mock_contigs[chr_name]
    else:
        zymo_log_map_df.loc[i, "genome"] = "Unknown"

3395063 3571140


In [8]:
zymo_log_map_df 

,read,chr,pos,read_length,effective_coverage,subread_passes,predicted_concordance,alignment_type,strand,alignment_mapq,...,concordance,gap_compressed_concordance,concordance_qv,mismatches,non_hp_ins,non_hp_del,hp_ins,hp_del,genome,Genome
0,ERR3152366.1317769,BS.pilon.polished.v3.ST170922,1,713,NaN,NaN,NaN,supplementary,+,60,...,0.919728,0.929849,10.95,29,3,10,5,12,NaN,Unknown
1,ERR3152366.1773054,BS.pilon.polished.v3.ST170922,1,1810,NaN,NaN,NaN,supplementary,+,60,...,0.930096,0.947798,11.55,33,21,32,13,32,NaN,Unknown
2,ERR3152366.730384,BS.pilon.polished.v3.ST170922,1,379,NaN,NaN,NaN,supplementary,-,60,...,0.910486,0.927083,10.48,10,10,7,3,5,NaN,Unknown
3,ERR3152366.150570,BS.pilon.polished.v3.ST170922,1,795,NaN,NaN,NaN,supplementary,-,60,...,0.855263,0.881628,8.39,48,19,19,13,22,NaN,Unknown
4,ERR3152366.2119173,BS.pilon.polished.v3.ST170922,1,5225,NaN,NaN,NaN,primary,+,60,...,0.911959,0.930389,10.55,82,52,30,23,53,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3571135,ERR3152366.1289031,Salmonella_enterica_chromosome,4735797,8826,NaN,NaN,NaN,primary,-,60,...,0.856313,0.886561,8.43,476,226,218,117,298,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,Unknown
3571136,ERR3152366.2420450,Salmonella_enterica_chromosome,4743787,13458,NaN,NaN,NaN,primary,-,60,...,0.897018,0.926093,9.87,410,223,315,130,362,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,Unknown
3571137,ERR3152366.2930035,Salmonella_enterica_chromosome,4748163,11635,NaN,NaN,NaN,primary,-,60,...,0.856073,0.888678,8.42,665,311,292,125,337,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,Unknown
3571138,ERR3152366.2387453,Staphylococcus_aureus_chromosome,1088305,7504,NaN,NaN,NaN,primary,+,60,...,0.786377,0.842707,6.70,535,551,242,137,216,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,Unknown


In [12]:
# Count zymo_log_map_df by genome and store as zymo_log_read_count_df
zymo_log_read_count_df = zymo_log_map_df.groupby("genome")["read"].nunique().reset_index()
zymo_log_read_count_df.columns = ["Genome", "num_reads"]
zymo_log_read_count_df


,Genome,num_reads
0,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,39883
1,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,206
2,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,17478
3,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,1881
4,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,178
5,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,3127744
6,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,177623
7,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,28277
8,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,1787
9,../../data/zymo/ZymoBIOMICS.STD.refseq.v2/Geno...,2


In [14]:
# basename of the Genome_file field of sylph_res_df
import os

sylph_recognized_genomes = [
    os.path.basename(genome) for genome in sylph_res_df["Genome_file"].unique()
]
# change the genome field of zymo_log_map_df to the basename of the genome field
zymo_log_read_count_df["Genome"] = zymo_log_read_count_df["Genome"].apply(lambda x: os.path.basename(x))
# the total number of reads that mapped to the recognized genomes in sylph_res_df
recognized_genome_read_count = zymo_log_read_count_df[zymo_log_read_count_df["Genome"].isin(sylph_recognized_genomes)]["num_reads"].sum()
recognized_genome_read_count

3377373

In [15]:
# Number of classified reads
3377373 / 3667480

0.9208974554735131

In [18]:
import pandas as pd

zymo_gut_map_df = pd.read_csv("../../output/zymo/SRR13128014.per_aln_stats.csv")
sylph_res_df = pd.read_csv("../../output/predict_unclassified_reads/SRR13128014_sylph_gt_u.tsv", sep="\t")

print(len(set(zymo_gut_map_df["read"])), len(zymo_gut_map_df))

for i in range(len(zymo_gut_map_df)):
    if zymo_gut_map_df.loc[i, "alignment_type"] != "primary":
        continue
    chr_name = zymo_gut_map_df.loc[i, "chr"]
    if chr_name in zymo_gut_contigs:
        zymo_gut_map_df.loc[i, "genome"] = zymo_gut_contigs[chr_name]
    else:
        zymo_gut_map_df.loc[i, "genome"] = "Unknown"

1978582 2032998


In [19]:
zymo_gut_read_count_df = zymo_gut_map_df.groupby("genome")["read"].nunique().reset_index()
zymo_gut_read_count_df.columns = ["Genome", "num_reads"]
zymo_gut_read_count_df

,Genome,num_reads
0,../../data/zymo/D6331.refseq/genomes/Akkermans...,39502
1,../../data/zymo/D6331.refseq/genomes/Bacteroid...,380947
2,../../data/zymo/D6331.refseq/genomes/Bifidobac...,38734
3,../../data/zymo/D6331.refseq/genomes/Candida_a...,4361
4,../../data/zymo/D6331.refseq/genomes/Clostridi...,52982
5,../../data/zymo/D6331.refseq/genomes/Enterococ...,15
6,../../data/zymo/D6331.refseq/genomes/Escherich...,68747
7,../../data/zymo/D6331.refseq/genomes/Escherich...,70828
8,../../data/zymo/D6331.refseq/genomes/Escherich...,64424
9,../../data/zymo/D6331.refseq/genomes/Escherich...,61453


In [20]:
# basename of the Genome_file field of sylph_res_df
import os

sylph_recognized_genomes = [
    os.path.basename(genome) for genome in sylph_res_df["Genome_file"].unique()
]
# change the genome field of zymo_log_map_df to the basename of the genome field
zymo_gut_read_count_df["Genome"] = zymo_gut_read_count_df["Genome"].apply(lambda x: os.path.basename(x))
# the total number of reads that mapped to the recognized genomes in sylph_res_df
recognized_genome_read_count = zymo_gut_read_count_df[zymo_gut_read_count_df["Genome"].isin(sylph_recognized_genomes)]["num_reads"].sum()
recognized_genome_read_count

1725357

In [ ]:
zymo_gut_read_count_df.

In [21]:
1725357 / 1978852

0.8718979489117933